In [2]:
import os
import re
import time
import ee
import numpy as np
import pandas as pd

# ==============================================================================
# 1. إعداد المشروع والاتصال بـ Google Earth Engine
# ==============================================================================
# استبدل القيمة بالأسفل بمعرّف مشروعك الإنجليزي الجديد في حال استمرار restricted mode
PROJECT_ID = "acoustic-gizmo-513018-a8"

if not PROJECT_ID.isascii() or " " in PROJECT_ID:
    raise ValueError("يجب أن يكون PROJECT_ID باللغة الإنجليزية وبدون مسافات (مثال: field-shift-egypt-441512).")

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# ==============================================================================
# 2. تعريف المراكز الزراعية الـ 14 في مصر
# ==============================================================================
HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94), ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67), ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00), ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18), ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84), ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75), ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69), ("Qena_Luxor", 25.69, 32.64),
]

hubs_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]), {"Governorate": name})
    for name, lat, lon in HUBS
])

# ==============================================================================
# 3. تجهيز مجموعات بيانات MODIS (NDVI, EVI, ET, PET)
# ==============================================================================
modis_veg = ee.ImageCollection("MODIS/061/MOD13Q1").select(["NDVI", "EVI"]).map(
    lambda img: img.multiply(0.0001).rename(["MODIS_NDVI", "MODIS_EVI"])
)
modis_et = ee.ImageCollection("MODIS/061/MOD16A2GF").select(["ET", "PET"]).map(
    lambda img: img.multiply(0.1).rename(["MODIS_ET_Actual_mm", "MODIS_PET_Potential_mm"])
)
BANDS = ["MODIS_NDVI", "MODIS_EVI", "MODIS_ET_Actual_mm", "MODIS_PET_Potential_mm"]


def safe_mean(col, band_names):
    dummy = ee.Image.constant([0] * len(band_names)).toFloat().rename(band_names).updateMask(ee.Image.constant(0))
    return col.map(lambda img: img.toFloat()).merge(ee.ImageCollection([dummy])).mean()


def get_info_with_retry(ee_obj, retries=5):
    for attempt in range(retries):
        try:
            return ee_obj.getInfo()
        except Exception as err:
            if attempt == retries - 1:
                raise
            wait_s = 15 * (attempt + 1)
            print(f"      ⏳ ضغط مؤقت على الخادم ({err}) -> انتظار {wait_s} ثانية...", flush=True)
            time.sleep(wait_s)


# ==============================================================================
# 4. بناء الشبكة الزمنية (365 فترة × 10 أيام) والسحب على دفعات خفيفة
# ==============================================================================
date_bins = pd.date_range("2016-01-01", "2025-12-31", freq="10D")
expected_dates = pd.to_datetime(date_bins[:-1])
intervals = [
    (date_bins[i].strftime("%Y-%m-%d"), date_bins[i + 1].strftime("%Y-%m-%d"))
    for i in range(len(date_bins) - 1)
]

BATCH_SIZE = 18  # 6 أشهر لكل دفعة لتقليل استهلاك الذاكرة
batches = [intervals[i:i + BATCH_SIZE] for i in range(0, len(intervals), BATCH_SIZE)]

records_dict = {}
print(f"📥 [5/6] بدء سحب بيانات NASA MODIS (NDVI, EVI, ET, PET) ({len(batches)} دفعة)...", flush=True)

for idx, batch in enumerate(batches, start=1):
    stack = ee.ImageCollection([
        safe_mean(modis_veg.filterDate(s, e), ["MODIS_NDVI", "MODIS_EVI"])
        .addBands(safe_mean(modis_et.filterDate(s, e), ["MODIS_ET_Actual_mm", "MODIS_PET_Potential_mm"]))
        .rename([f"{s}_{b}" for b in BANDS])
        for s, e in batch
    ]).toBands()

    fc_batch = stack.reduceRegions(
        collection=hubs_fc,
        reducer=ee.Reducer.first(),
        scale=500,
        tileScale=4
    )
    res = get_info_with_retry(fc_batch)

    for feat in res["features"]:
        props = feat["properties"]
        gov = props["Governorate"]
        for k, val in props.items():
            m = re.match(r"^(?:\d+_)?(\d{4}-\d{2}-\d{2})_(.+)$", k)
            if m:
                records_dict.setdefault((gov, m.group(1)), {})[m.group(2)] = val

    print(f"   ✓ MODIS: اكتملت الدفعة {idx}/{len(batches)} ({batch[0][0]} إلى {batch[-1][1]})", flush=True)
    time.sleep(1.5)

# ==============================================================================
# 5. المطابقة على الشبكة الموحدة، التحويل الرقمي، وحفظ الملف
# ==============================================================================
df_raw = pd.DataFrame([{"Governorate": gov, "Date": dt, **vals} for (gov, dt), vals in records_dict.items()])
if not df_raw.empty:
    df_raw["Date"] = pd.to_datetime(df_raw["Date"])
else:
    df_raw = pd.DataFrame(columns=["Governorate", "Date"] + BANDS)

grid = pd.MultiIndex.from_product(
    [[h[0] for h in HUBS], expected_dates],
    names=["Governorate", "Date"]
).to_frame().reset_index(drop=True)

out = pd.merge(grid, df_raw, on=["Governorate", "Date"], how="left").sort_values(["Governorate", "Date"]).reset_index(drop=True)

# ضمان وجود كافة الأعمدة وتحويلها صراحةً إلى أرقام (float64) لمنع خطأ No objects to concatenate
for col in BANDS:
    if col not in out.columns:
        out[col] = np.nan
    out[col] = pd.to_numeric(out[col], errors="coerce")

for col in BANDS:
    out[col] = out.groupby("Governorate")[col].transform(lambda g: g.interpolate("linear").ffill().bfill())

# سد أي عمود فارغ بالكامل بقيمة 0 احتياطياً
out[BANDS] = out[BANDS].fillna(0.0)
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

output_path = r"d:\vscode\NASA\DATA\05_MODIS_NDVI_EVI_ET.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
out.to_csv(output_path, index=False)

print(f"\n✅ اكتمل الملف الخامس بنجاح: {output_path}")
print(f"📐 الأبعاد: {out.shape[0]} صف × {out.shape[1]} عمود | القيم المفقودة: {out.isnull().sum().sum()}")

📥 [5/6] بدء سحب بيانات NASA MODIS (NDVI, EVI, ET, PET) (21 دفعة)...
   ✓ MODIS: اكتملت الدفعة 1/21 (2016-01-01 إلى 2016-06-29)


C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


   ✓ MODIS: اكتملت الدفعة 2/21 (2016-06-29 إلى 2016-12-26)
   ✓ MODIS: اكتملت الدفعة 3/21 (2016-12-26 إلى 2017-06-24)
   ✓ MODIS: اكتملت الدفعة 4/21 (2017-06-24 إلى 2017-12-21)
   ✓ MODIS: اكتملت الدفعة 5/21 (2017-12-21 إلى 2018-06-19)
   ✓ MODIS: اكتملت الدفعة 6/21 (2018-06-19 إلى 2018-12-16)
   ✓ MODIS: اكتملت الدفعة 7/21 (2018-12-16 إلى 2019-06-14)
   ✓ MODIS: اكتملت الدفعة 8/21 (2019-06-14 إلى 2019-12-11)
   ✓ MODIS: اكتملت الدفعة 9/21 (2019-12-11 إلى 2020-06-08)
   ✓ MODIS: اكتملت الدفعة 10/21 (2020-06-08 إلى 2020-12-05)
   ✓ MODIS: اكتملت الدفعة 11/21 (2020-12-05 إلى 2021-06-03)
   ✓ MODIS: اكتملت الدفعة 12/21 (2021-06-03 إلى 2021-11-30)
   ✓ MODIS: اكتملت الدفعة 13/21 (2021-11-30 إلى 2022-05-29)
   ✓ MODIS: اكتملت الدفعة 14/21 (2022-05-29 إلى 2022-11-25)
   ✓ MODIS: اكتملت الدفعة 15/21 (2022-11-25 إلى 2023-05-24)
   ✓ MODIS: اكتملت الدفعة 16/21 (2023-05-24 إلى 2023-11-20)
   ✓ MODIS: اكتملت الدفعة 17/21 (2023-11-20 إلى 2024-05-18)
   ✓ MODIS: اكتملت الدفعة 18/21 (2024-05-18 إلى 